In [7]:
# import packages
import os

import pandas as pd

from calendar import monthrange
from datetime import datetime, timedelta

from azureml.core import Dataset, Datastore, Workspace
from azureml.opendatasets import NoaaIsdWeather

In [8]:
ws = Workspace.from_config()
dstore = ws.get_default_datastore()

In [9]:
ws

Workspace.create(name='DataDrift-Prod-EUAP', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG')

In [10]:
%%time 

target_years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

for year in target_years:
    for month in range(1, 12+1):
        path = 'weather-data/{}/{:02d}/'.format(year, month)
        
        try:  
            start = datetime(year, month, 1)
            end   = datetime(year, month, monthrange(year, month)[1]) + timedelta(days=1)
            isd   = NoaaIsdWeather(start, end).to_pandas_dataframe()
            isd   = isd[isd['stationName'].str.contains('FLORIDA|WASHINGTON|TEXAS|MIAMI|SEATTLE|HOUSTON', regex=True, na=False)]
            
            os.makedirs(path, exist_ok=True)
            isd.to_parquet(path + 'data.parquet')
        except Exception as e:
            print('Month {} in year {} likely has no data.\n'.format(month, year))
            print('Exception: {}'.format(e))

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2010/month=1/', '/year=2010/month=2/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2010/month=1/part-00009-tid-2760641154746282667-d6a50598-0478-455f-9217-ce4afb96a9bb-36.c000.snappy.parquet under container isdweatherdatacontainer
Reading ISDWeather/year=2010/month=2/part-00011-tid-2760641154746282667-d6a50598-0478-455f-9217-ce4afb96a9bb-38.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=101898.4 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=101922.07 [ms]
ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2010/month=2/', '/year=2010/month=3/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2010/month=

In [11]:
# uploads everything in the folder 'data' to the datastore under a container named 'weather-data'
dstore.upload('weather-data', 'weather-data')

Uploading an estimated of 120 files
Uploading weather-data/2010/01/data.parquet
Uploading weather-data/2010/02/data.parquet
Uploading weather-data/2010/03/data.parquet
Uploading weather-data/2010/04/data.parquet
Uploading weather-data/2010/05/data.parquet
Uploading weather-data/2010/06/data.parquet
Uploading weather-data/2010/07/data.parquet
Uploading weather-data/2010/08/data.parquet
Uploading weather-data/2010/09/data.parquet
Uploading weather-data/2010/10/data.parquet
Uploading weather-data/2010/11/data.parquet
Uploading weather-data/2010/12/data.parquet
Uploading weather-data/2011/01/data.parquet
Uploading weather-data/2011/02/data.parquet
Uploading weather-data/2011/03/data.parquet
Uploading weather-data/2011/04/data.parquet
Uploading weather-data/2011/05/data.parquet
Uploading weather-data/2011/06/data.parquet
Uploading weather-data/2011/07/data.parquet
Uploading weather-data/2011/08/data.parquet
Uploading weather-data/2011/09/data.parquet
Uploading weather-data/2011/10/data.parq

$AZUREML_DATAREFERENCE_ff1a0dc45bc84cca92468feb20d65e55

In [24]:
for year in os.listdir('weather-data'):
    for month in os.listdir('weather-data/{}'.format(year)):
        for filename in os.listdir('weather-data/{}/{}'.format(year, month)):
            print('weather-data/{}/{}/{}'.format(year, month, filename))

weather-data/2010/01/data.parquet
weather-data/2010/02/data.parquet
weather-data/2010/03/data.parquet
weather-data/2010/04/data.parquet
weather-data/2010/05/data.parquet
weather-data/2010/06/data.parquet
weather-data/2010/07/data.parquet
weather-data/2010/08/data.parquet
weather-data/2010/09/data.parquet
weather-data/2010/10/data.parquet
weather-data/2010/11/data.parquet
weather-data/2010/12/data.parquet
weather-data/2011/01/data.parquet
weather-data/2011/02/data.parquet
weather-data/2011/03/data.parquet
weather-data/2011/04/data.parquet
weather-data/2011/05/data.parquet
weather-data/2011/06/data.parquet
weather-data/2011/07/data.parquet
weather-data/2011/08/data.parquet
weather-data/2011/09/data.parquet
weather-data/2011/10/data.parquet
weather-data/2011/11/data.parquet
weather-data/2011/12/data.parquet
weather-data/2012/01/data.parquet
weather-data/2012/02/data.parquet
weather-data/2012/03/data.parquet
weather-data/2012/04/data.parquet
weather-data/2012/05/data.parquet
weather-data/2

In [22]:
from glob import glob

glob('/weather-data/*/*/*/data.parquet')

[]